# Booze ’R’ Us — Raw Liquor Sales Data Build

**Purpose:** combine the raw Iowa Liquor Sales export files into a single
deduplicated parquet file, `liquor_2022_2026.parquet`. This is the raw
transaction-level dataset that `02_build_city_category_month_data.ipynb`
aggregates into the modeling dataset used by the forecast analysis.

**Data source:** Iowa Liquor Sales retail transaction data, from Iowa's
open data portal (`idh-be.iowa.gov`). Retailer purchase orders are exported
there one file per year; this notebook expects the 2022–2026 yearly exports
already downloaded as zip files. See the top-level `README.md` for exactly
where to get these files and how to lay them out locally, since the raw
data itself is not included in this submission.

In [1]:
import duckdb
import pandas as pd

# Step 1 — Combine and Deduplicate the Yearly Exports

Each yearly export is a zip file containing the sale records as JSON part
files. We extract one year at a time, load it into DuckDB, and remove exact
duplicate rows before moving to the next year. Deduplicating year by year
(rather than loading all five years into memory at once) keeps memory use
manageable and makes it easy to see how many duplicates came from which
year's export.

A direct CSV download from the portal's API was tried first
(`https://idh-be.iowa.gov/api/v1/datasets/<id>/rows.csv`) but the endpoint
did not reliably return a clean CSV. Downloading each year's export as a
zip file directly from the portal and processing it locally, as below,
avoided that issue entirely.

In [2]:
import zipfile
import glob
import os
import re
import shutil

# Each yearly export is a zip named iowa_liquor_sales_<year>_<id>_rows.zip,
# expected under this folder (see README.md for how to obtain them).
raw_zip_dir = "liquor_2022_2026"
source_zips = sorted(glob.glob(f"{raw_zip_dir}/*.zip"))
if not source_zips:
    raise FileNotFoundError(f"No zip files found under '{raw_zip_dir}/'.")
print(f"Using {len(source_zips)} source zip(s): {source_zips}")

work_dir = "liquor_2022_2026_extracted"
tmp_dir = os.path.join(work_dir, "_duckdb_tmp")
shutil.rmtree(work_dir, ignore_errors=True)
os.makedirs(tmp_dir, exist_ok=True)

# Give DuckDB an explicit disk-backed spill directory and a memory cap so a
# large DISTINCT over several million rows spills to disk cleanly instead
# of exhausting memory on a laptop-scale machine.
con = duckdb.connect()
con.execute(f"SET temp_directory='{tmp_dir}'")
con.execute("SET memory_limit='6GB'")
con.execute("SET preserve_insertion_order=false")

# Process one year at a time: extract, load, and drop exact-duplicate rows
# before combining with the other years. Each year's extraction directory
# is wiped first so a rerun can't pick up leftover files from a prior run.
per_year_parquets = []
before_total = 0
after_total = 0

for sz in source_zips:
    year_match = re.search(r"(\d{4})", os.path.basename(sz))
    year_label = year_match.group(1) if year_match else os.path.basename(sz)

    year_extract_dir = os.path.join(work_dir, year_label)
    os.makedirs(year_extract_dir, exist_ok=True)

    print(f"[{year_label}] extracting {sz} ...")
    with zipfile.ZipFile(sz, "r") as zip_ref:
        zip_ref.extractall(year_extract_dir)

    json_files = glob.glob(f"{year_extract_dir}/**/*.json", recursive=True)
    if not json_files:
        raise FileNotFoundError(f"No JSON files found for {sz} under {year_extract_dir}")

    con.execute("DROP TABLE IF EXISTS year_raw")
    con.execute(
        "CREATE TABLE year_raw AS SELECT * FROM read_json_auto(?, union_by_name=true)",
        [json_files],
    )
    n_raw = con.execute("SELECT COUNT(*) FROM year_raw").fetchone()[0]

    year_out = os.path.join(work_dir, f"liquor_{year_label}_dedup.parquet")
    con.execute(f"COPY (SELECT DISTINCT * FROM year_raw) TO '{year_out}' (FORMAT PARQUET)")
    n_dedup = con.execute(f"SELECT COUNT(*) FROM '{year_out}'").fetchone()[0]
    print(f"[{year_label}] {n_raw:,} raw -> {n_dedup:,} deduped (removed {n_raw - n_dedup:,})")

    before_total += n_raw
    after_total += n_dedup
    per_year_parquets.append(year_out)

    con.execute("DROP TABLE year_raw")
    shutil.rmtree(year_extract_dir, ignore_errors=True)  # free disk as we go

print(f"\nTotal raw rows across all years: {before_total:,}")
print(f"Total rows after per-year dedup: {after_total:,}")
print(f"Total exact duplicates removed:  {before_total - after_total:,}\n")

# Union the per-year deduped files and dedup once more across year
# boundaries (cheap now that the data is already shrunk).
out_path = "liquor_2022_2026.parquet"
tmp_path = "liquor_2022_2026_rebuild_tmp.parquet"

con.execute(
    f"""
    COPY (
        SELECT DISTINCT * FROM read_parquet({per_year_parquets}, union_by_name=true)
    )
    TO '{tmp_path}'
    (FORMAT PARQUET)
    """
)
final_count = con.execute(f"SELECT COUNT(*) FROM read_parquet('{tmp_path}')").fetchone()[0]
print(f"Final row count: {final_count:,} (cross-year duplicates removed: {after_total - final_count:,})")

os.replace(tmp_path, out_path)
print(f"Done! Saved {out_path}")

shutil.rmtree(work_dir, ignore_errors=True)

Using 5 source zip(s): ['liquor_2022_2026/iowa_liquor_sales_2022_1259_rows.zip', 'liquor_2022_2026/iowa_liquor_sales_2023_1260_rows.zip', 'liquor_2022_2026/iowa_liquor_sales_2024_1261_rows.zip', 'liquor_2022_2026/iowa_liquor_sales_2025_1262_rows.zip', 'liquor_2022_2026/iowa_liquor_sales_2026_1263_rows.zip']
[2022] extracting liquor_2022_2026/iowa_liquor_sales_2022_1259_rows.zip ...


[2022] 3,157,182 raw -> 2,564,565 deduped (removed 592,617)
[2023] extracting liquor_2022_2026/iowa_liquor_sales_2023_1260_rows.zip ...


[2023] 2,639,557 raw -> 2,639,557 deduped (removed 0)
[2024] extracting liquor_2022_2026/iowa_liquor_sales_2024_1261_rows.zip ...


[2024] 2,590,975 raw -> 2,590,975 deduped (removed 0)
[2025] extracting liquor_2022_2026/iowa_liquor_sales_2025_1262_rows.zip ...


[2025] 2,907,817 raw -> 2,490,613 deduped (removed 417,204)
[2026] extracting liquor_2022_2026/iowa_liquor_sales_2026_1263_rows.zip ...


[2026] 1,968,354 raw -> 1,604,194 deduped (removed 364,160)

Total raw rows across all years: 13,263,885
Total rows after per-year dedup: 11,889,904
Total exact duplicates removed:  1,373,981



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Final row count: 11,889,904 (cross-year duplicates removed: 0)
Done! Saved liquor_2022_2026.parquet


# Step 2 — Inspect the Combined Schema

A quick sanity check on the combined file before it is used downstream:
confirm the column names and types match what
`02_build_city_category_month_data.ipynb` expects
(`ordered_on`, `store_city`, `category_name`, `sales_bottles`,
`sales_liters`, `sales_dollars`).

In [3]:
print(con.execute("DESCRIBE SELECT * FROM 'liquor_2022_2026.parquet'").df())
con.execute("SELECT * FROM 'liquor_2022_2026.parquet' LIMIT 3").df()

            column_name column_type null   key default extra
0            invoice_id     VARCHAR  YES  None    None  None
1            ordered_on        DATE  YES  None    None  None
2              store_no     VARCHAR  YES  None    None  None
3            store_name     VARCHAR  YES  None    None  None
4         store_address     VARCHAR  YES  None    None  None
5            store_city     VARCHAR  YES  None    None  None
6        store_zip_code     VARCHAR  YES  None    None  None
7      county_fips_code     VARCHAR  YES  None    None  None
8           county_name     VARCHAR  YES  None    None  None
9         category_code     VARCHAR  YES  None    None  None
10        category_name     VARCHAR  YES  None    None  None
11        vendor_number     VARCHAR  YES  None    None  None
12          vendor_name     VARCHAR  YES  None    None  None
13              item_no     VARCHAR  YES  None    None  None
14              im_desc     VARCHAR  YES  None    None  None
15                 pack 

,invoice_id,ordered_on,store_no,store_name,store_address,store_city,store_zip_code,county_fips_code,county_name,category_code,...,item_no,im_desc,pack,bottle_volume_ml,sales_bottles,sales_dollars,sales_liters,sales_gallons,state_bottle_cost,state_bottle_retail
0,INV-46778100010,2022-04-22,2602,HY-VEE FOOD STORE / WEBSTER CITY,823 2ND ST,WEBSTER CITY,50595,19079,HAMILTON,1011200,...,19061,JIM BEAM MINI,12,50,1,10.50,0.05,0.01,NaN,NaN
1,INV-48062400060,2022-06-04,3592,WAL-MART 0886 / FORT DODGE,3036 1ST AVE SOUTH,FORT DODGE,50501,19083,HARDIN,1022200,...,88371,TEREMANA BLANCO TEQUILA,6,750,6,134.46,4.50,1.18,NaN,NaN
2,INV-46900800001,2022-04-27,2285,JOHN'S GROCERY,401 EAST MARKET ST,IOWA CITY,52240,19103,JOHNSON,1701100,...,27617,WHITE DOG WHEAT,12,375,1,13.50,0.37,0.09,NaN,NaN
